In [1]:
import numpy as np
import pandas as pd 
from subprocess import check_output
from gensim.models import Word2Vec

from nltk.tokenize import RegexpTokenizer
from nltk import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.linear_model import LogisticRegression

alpha_tokenizer = RegexpTokenizer('[A-Za-z]\w+')
lemmatizer = WordNetLemmatizer()
stop = stopwords.words('english')


In [2]:
train = pd.read_csv('../input/train.csv')
test = pd.read_csv('../input/test.csv')


In [3]:
data = [[lemmatizer.lemmatize(word.lower()) for word in alpha_tokenizer.tokenize(sent) if word.lower() not in stop] for sent in train.text.values]


In [4]:
NUM_FEATURES = 200

model = Word2Vec(data, min_count=3, size=NUM_FEATURES, window=5, sg=1, alpha=1e-4, workers=4)


TypeError: Word2Vec.__init__() got an unexpected keyword argument 'size'

In [5]:
len(model.wv.vocab)


NameError: name 'model' is not defined

In [6]:
model.most_similar('raven')


NameError: name 'model' is not defined

In [7]:
def get_feature_vec(tokens, num_features, model):
    featureVec = np.zeros(shape=(1, num_features), dtype='float32')
    missed = 0
    for word in tokens:
        try:
            featureVec = np.add(featureVec, model[word])
        except KeyError:
            missed += 1
            pass
    if len(tokens) - missed == 0:
        return np.zeros(shape=(num_features), dtype='float32')
    return np.divide(featureVec, len(tokens) - missed).squeeze()


In [8]:
vectors = []
for i in train.text.values:
    vectors.append(get_feature_vec([lemmatizer.lemmatize(word.lower()) for word in alpha_tokenizer.tokenize(i) if word.lower() not in stop], NUM_FEATURES, model))


NameError: name 'model' is not defined

In [9]:
train['author'] = train['author'].map({'EAP': 0, 'HPL': 1, 'MWS': 2})


In [10]:
estimator = LogisticRegression(C=1)
estimator.fit(np.array(vectors), train.author.values);


ValueError: Expected 2D array, got 1D array instead:
array=[].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [11]:
test_vectors = []
for i in test.text.values:
    test_vectors.append(get_feature_vec([lemmatizer.lemmatize(word.lower()) for word in alpha_tokenizer.tokenize(i) if word.lower() not in stop], NUM_FEATURES, model))


NameError: name 'model' is not defined

In [12]:
probs = estimator.predict_proba(test_vectors)


NotFittedError: This LogisticRegression instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [13]:
author = pd.DataFrame(probs)

final = pd.DataFrame()
final['id'] = test.id
final['EAP'] = author[0]
final['HPL'] = author[1]
final['MWS'] = author[2]
final.to_csv('submission.csv', sep=',',index=False)


NameError: name 'probs' is not defined